# J1-03 · StateGraph pas à pas : état, reducers, arêtes conditionnelles

**Formation « Orchestration d'agents & Monitoring : LangGraph + Langfuse (stack 100 % locale) »**
Jour 1 · créneau 13h00–14h30 · ⏱️ ~1h30 (dont 50 min de pratique)

| | |
|---|---|
| 🎯 **Objectifs** | 1. Construire un `StateGraph` : état, nœuds, arêtes, arêtes conditionnelles · 2. Comprendre les **reducers** (écraser, concaténer, ajouter) · 3. Exécuter avec `invoke`, `invoke(version="v2")` et `stream` · 4. Redessiner l'agent de J1-02 avec `ToolNode` + `tools_condition` · 5. Maîtriser `recursion_limit` |
| 🧱 **Fil rouge** | Le tri des tickets de J1-01 devient un **graphe** ; l'agent de J1-02 devient un graphe **explicite** |
| ✅ **Prérequis** | J1-01 et J1-02 terminés |
| 📦 **Versions** | `langgraph 1.2` · `langchain 1.4` · `langfuse 4.15`, figées par `uv.lock` |

## 1. Mise en route

### [Code] Le socle du fil rouge : `src/helpdesk_ops`

Le code écrit et expliqué en J1-01 (lecture de `.env`, masquage des données personnelles, client Langfuse, création du modèle) est rangé dans le paquet **`helpdesk_ops`**, installé par `uv sync`. Chaque notebook reste **autonome** : la cellule suivante suffit.

> 💡 **Le « Tips » du Formateur :** le paquet est installé en mode *éditable* : une modification dans `src/helpdesk_ops` est prise en compte après *Kernel → Restart*. Le client Langfuse est un singleton par clé publique : en cas de doute, redémarrez le kernel.

In [ ]:
import os

from langfuse import observe, propagate_attributes

from helpdesk_ops.config import creer_llm
from helpdesk_ops.observabilite import init_observabilite
from helpdesk_ops.texte import sans_accents

langfuse, langfuse_handler = init_observabilite()   # .env chargé, masquage PII actif, clés vérifiées
llm = creer_llm("agent")                             # MODEL_AGENT, temperature=0, num_ctx=16384
print(f"✅ Langfuse : {os.environ['LANGFUSE_BASE_URL']} | modèle : {llm.model}")

## 2. Les 5 concepts de LangGraph

### [Théorie] Un organigramme exécutable

*Analogie : l'organigramme du traitement d'un ticket.* Chaque **case** est une fonction, chaque **flèche** une ligne de code, et le **dossier** qui circule de case en case est l'état.

| Concept | Ce que c'est | Code |
|---|---|---|
| **State** | le dossier partagé (`TypedDict`) | `class TicketState(TypedDict): ...` |
| **Nœud** | une fonction `(state) -> dict` qui renvoie une **mise à jour partielle** | `builder.add_node("classer", classer)` |
| **Arête** | « a puis b » | `builder.add_edge("a", "b")` |
| **Arête conditionnelle** | une fonction de routage qui renvoie le nom du nœud suivant | `builder.add_conditional_edges("classer", router)` |
| **START / END** | entrée et sortie ; `compile()` valide et rend le graphe exécutable | `builder.compile()` |

> 💡 **Le « Tips » du Formateur :** un nœud renvoie **uniquement les clés qu'il modifie**. Jamais l'état complet, jamais un état muté sur place (`state["x"] = …`). LangGraph fusionne les mises à jour lui-même.

In [ ]:
from typing import Literal, TypedDict

from langgraph.graph import END, START, StateGraph


class TicketState(TypedDict):
    ticket: str
    categorie: str
    file: str


def classer(state: TicketState) -> dict:
    # Règle Python volontairement simple : on isole la mécanique du graphe. Le LLM arrive en section 5.
    texte = sans_accents(state["ticket"]).lower()
    return {"categorie": "reseau" if "vpn" in texte or "reseau" in texte else "logiciel"}   # mise à jour PARTIELLE


def router(state: TicketState) -> Literal["n2_infra", "n1"]:   # Literal : sert au dessin du graphe
    return "n2_infra" if state["categorie"] == "reseau" else "n1"


def n2_infra(state: TicketState) -> dict:
    return {"file": "N2-infra"}


def n1(state: TicketState) -> dict:
    return {"file": "N1"}


builder = StateGraph(TicketState)
builder.add_node("classer", classer)
builder.add_node("n2_infra", n2_infra)
builder.add_node("n1", n1)
builder.add_edge(START, "classer")
builder.add_conditional_edges("classer", router)   # la décision
builder.add_edge("n2_infra", END)
builder.add_edge("n1", END)
graphe = builder.compile()

print(graphe.invoke({"ticket": "VPN coupé depuis ce matin"}))
print(graphe.invoke({"ticket": "Excel plante à l'ouverture"}))

### [Code] Visualiser le graphe, sans quitter la machine

`draw_mermaid_png()` appelle par défaut l'API web `mermaid.ink` : **un flux sortant**, contraire au contrat 100 % local. On préfère le texte Mermaid, que JupyterLab 4 affiche nativement, ou l'ASCII.

In [ ]:
from IPython.display import Markdown, display

display(Markdown(f"```mermaid\n{graphe.get_graph().draw_mermaid()}\n```"))
print(graphe.get_graph().draw_ascii())   # repli hors JupyterLab (nécessite grandalf, installé)

**👀 Ce que vous devez observer :** `classer` a **deux** arêtes en pointillés (conditionnelles) vers `n2_infra` et `n1`. Sans l'annotation `Literal[...]` sur `router`, LangGraph ne saurait pas les dessiner.

## 3. Exécuter : `invoke`, `invoke(version="v2")`, `stream`

### [Théorie] Trois façons de lancer un graphe

- `invoke` renvoie l'**état final** (un `dict`) ;
- `invoke(..., version="v2")` (LangGraph ≥ 1.1) renvoie un objet typé avec `.value` (l'état) et `.interrupts` (les pauses de validation humaine, J2) ;
- `stream(..., stream_mode=...)` émet le graphe **étape par étape** :

| `stream_mode` | Vous recevez | Usage |
|---|---|---|
| `"updates"` | la mise à jour renvoyée par chaque nœud | déboguer, journaliser |
| `"values"` | l'état complet après chaque étape | afficher un état qui évolue |
| `"messages"` | les tokens du LLM au fil de l'eau | interface de chat (Streamlit, J5) |

In [ ]:
sortie = graphe.invoke({"ticket": "VPN coupé"}, version="v2")
print("v2      :", type(sortie).__name__, "| value =", sortie.value, "| interrupts =", sortie.interrupts)

print("\nupdates :")
for maj in graphe.stream({"ticket": "Excel plante"}, stream_mode="updates"):
    print("  ", maj)

print("\nvalues  :")
for etat in graphe.stream({"ticket": "Excel plante"}, stream_mode="values"):
    print("  ", etat)

**👀 Ce que vous devez observer :** `updates` montre **qui** a modifié **quoi** (`{'classer': {'categorie': 'logiciel'}}`), `values` montre l'état qui se remplit. Le mode `messages` arrive en section 6, avec un vrai LLM.

## 4. Les reducers : écraser, concaténer ou ajouter

### [Théorie] Qui décide comment une mise à jour est fusionnée ?

Par défaut, une clé est **écrasée** par la dernière mise à jour. Un **reducer**, déclaré avec `Annotated[type, reducer]`, change la règle :

| Déclaration | Effet | Exemple |
|---|---|---|
| `journal: list[str]` | **écrase** | l'historique disparaît |
| `journal: Annotated[list[str], operator.add]` | **concatène** | un journal d'audit |
| `messages: Annotated[list, add_messages]` | **ajoute**, ou **remplace** un message de même `id` | une conversation |

*Analogie : le dossier partagé.* Sans reducer, chaque service **remplace** le classeur ; avec `operator.add`, il **ajoute sa feuille**.

> 💡 **Le « Tips » du Formateur :** `MessagesState` est simplement un `TypedDict` qui contient déjà `messages: Annotated[list, add_messages]`. C'est la base de tout agent conversationnel.

In [ ]:
import operator
from typing import Annotated


class SansReducer(TypedDict):
    journal: list[str]


class AvecReducer(TypedDict):
    journal: Annotated[list[str], operator.add]


def etape_a(state) -> dict:
    return {"journal": ["A : ticket reçu"]}


def etape_b(state) -> dict:
    return {"journal": ["B : ticket classé"]}


def construire(schema):
    b = StateGraph(schema)
    b.add_node("a", etape_a)
    b.add_node("b", etape_b)
    b.add_edge(START, "a")
    b.add_edge("a", "b")
    b.add_edge("b", END)
    return b.compile()


print("Sans reducer :", construire(SansReducer).invoke({"journal": ["début"]}))
print("Avec reducer :", construire(AvecReducer).invoke({"journal": ["début"]}))

**👀 Ce que vous devez observer :** sans reducer, il ne reste que `['B : ticket classé']` ; avec `operator.add`, les trois entrées sont conservées.

### [Code] Pourquoi c'est vital en parallèle

Si deux nœuds écrivent la **même clé au même moment** (branches parallèles, J3), LangGraph ne peut pas choisir : sans reducer, il lève une erreur.

In [ ]:
from langgraph.errors import InvalidUpdateError


def parallele(schema):
    b = StateGraph(schema)
    b.add_node("a", etape_a)
    b.add_node("b", etape_b)
    b.add_edge(START, "a")   # START -> a et START -> b : les deux s'exécutent dans la même étape
    b.add_edge(START, "b")
    b.add_edge("a", END)
    b.add_edge("b", END)
    return b.compile()


try:
    parallele(SansReducer).invoke({"journal": []})
except InvalidUpdateError as erreur:
    print("Sans reducer :", type(erreur).__name__, "→", str(erreur).splitlines()[0])
print("Avec reducer :", parallele(AvecReducer).invoke({"journal": []}))

In [ ]:
from langchain_core.messages import AIMessage, HumanMessage
from langgraph.graph.message import add_messages

historique = [HumanMessage("Mon VPN est coupé", id="m1"), AIMessage("Je vérifie srv-vpn-01…", id="m2")]

# add_messages AJOUTE un nouveau message…
print([m.content for m in add_messages(historique, [AIMessage("srv-vpn-01 est en ligne.", id="m3")])])
# …et REMPLACE un message qui porte le même id (utile pour corriger ou caviarder un message)
print([m.content for m in add_messages(historique, [AIMessage("Je vérifie le VPN.", id="m2")])])

## 5. Le LLM entre dans le graphe : un tri de tickets tracé

### [Théorie] Un nœud peut appeler un LLM comme n'importe quelle fonction

On remplace la règle `"vpn" in texte` par le classifieur LCEL de J1-01. Le graphe ne change pas : seule la fonction du nœud change. Pour le tracing, on passe le `CallbackHandler` **une fois**, dans le `config` de l'appel au graphe : il suit chaque nœud et chaque appel LLM imbriqué.

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

CATEGORIES = ["reseau", "materiel", "logiciel", "acces", "autre"]
classifieur = ChatPromptTemplate.from_messages([
    ("system",
     "Tu es l'agent de tri N1 du help-desk de NordTech SA. "
     "Classe le ticket dans UNE catégorie parmi : {categories}. "
     "Réponds uniquement par le nom de la catégorie, en minuscules, sans ponctuation."),
    ("human", "Ticket : {ticket}"),
]) | llm | StrOutputParser()


def normaliser_categorie(brute: str) -> str:
    nettoyee = sans_accents(brute).strip().lower().rstrip(".")
    return next((c for c in CATEGORIES if c in nettoyee), "autre")


def classer_llm(state: TicketState) -> dict:
    brute = classifieur.invoke({"ticket": state["ticket"], "categories": ", ".join(CATEGORIES)})
    return {"categorie": normaliser_categorie(brute)}


def router_llm(state: TicketState) -> Literal["n2_infra", "n1"]:
    return "n2_infra" if state["categorie"] in ("reseau", "materiel") else "n1"


tri = StateGraph(TicketState)
tri.add_node("classer", classer_llm)
tri.add_node("n2_infra", n2_infra)
tri.add_node("n1", n1)
tri.add_edge(START, "classer")
tri.add_conditional_edges("classer", router_llm)
tri.add_edge("n2_infra", END)
tri.add_edge("n1", END)
tri = tri.compile()

with propagate_attributes(session_id="j1-03-tri", tags=["jour-1", "stategraph"], version="tri-graphe-v1"):
    for ticket in ("Le Wi-Fi de la salle B coupe sans arrêt", "Je n'ai plus accès au dossier Compta", "Outlook plante"):
        etat = tri.invoke({"ticket": ticket}, config={"callbacks": [langfuse_handler], "run_name": "tri-ticket"})
        print(f"{etat['categorie']:<9} → {etat['file']:<9} ← {ticket}")
langfuse.flush()

**👀 Ce que vous devez observer :** dans Langfuse, chaque trace `tri-ticket` est un **arbre** qui reproduit le graphe : `classer` (avec la generation du LLM dedans), puis `n2_infra` ou `n1`. **Le code devient un schéma, et le schéma devient une trace.**

## 6. L'agent de J1-02 en graphe explicite : `ToolNode` + `tools_condition`

### [Théorie] La boucle ReAct, dessinée

| Boucle à la main (J1-02) | Graphe explicite |
|---|---|
| `messages = [...]` | `MessagesState` (reducer `add_messages`) |
| `reponse = llm_outils.invoke(messages)` | nœud `agent` |
| `if not reponse.tool_calls: break` | arête conditionnelle `tools_condition` (→ `tools` ou `END`) |
| `for appel in reponse.tool_calls: ...` | nœud `tools` = `ToolNode([...])` |
| `for tour in range(5)` | `recursion_limit` |

In [ ]:
from langchain_core.messages import SystemMessage
from langchain_core.tools import tool
from langgraph.graph import MessagesState
from langgraph.prebuilt import ToolNode, tools_condition

INVENTAIRE = {"srv-mail-01": "en ligne", "srv-erp-02": "hors ligne", "srv-vpn-01": "en ligne"}


@tool(parse_docstring=True)
def statut_serveur(nom: str) -> str:
    """Retourne le statut (en ligne, hors ligne) d'un serveur de l'inventaire NordTech.

    Args:
        nom: nom d'hôte exact en minuscules, par exemple 'srv-mail-01'
    """
    return INVENTAIRE.get(nom.strip().lower(), f"inconnu : '{nom}' n'est pas dans l'inventaire")


OUTILS = [statut_serveur]
llm_outils = llm.bind_tools(OUTILS)
SYSTEME = SystemMessage("Tu es l'agent support N1 de NordTech SA. Réponds en français, brièvement. Utilise les outils ; n'invente rien.")


def agent(state: MessagesState) -> dict:
    # Le prompt système est ajouté à l'appel, pas stocké dans l'état : l'historique reste propre
    return {"messages": [llm_outils.invoke([SYSTEME, *state["messages"]])]}


g = StateGraph(MessagesState)
g.add_node("agent", agent)
g.add_node("tools", ToolNode(OUTILS))
g.add_edge(START, "agent")
g.add_conditional_edges("agent", tools_condition)   # -> "tools" ou END
g.add_edge("tools", "agent")                        # la boucle ReAct, dessinée
support = g.compile()

display(Markdown(f"```mermaid\n{support.get_graph().draw_mermaid()}\n```"))

In [ ]:
with propagate_attributes(session_id="j1-03-agent", tags=["jour-1", "agent-graphe"]):
    resultat = support.invoke(
        {"messages": [HumanMessage("srv-erp-02 et srv-mail-01 sont-ils en ligne ?")]},
        config={"callbacks": [langfuse_handler], "run_name": "agent-graphe", "recursion_limit": 10},
    )
for m in resultat["messages"]:
    print(f"{type(m).__name__:<13} {m.content[:80]!r} {getattr(m, 'tool_calls', '') or ''}")
langfuse.flush()

**👀 Ce que vous devez observer :** la même séquence de messages qu'avec `create_agent`, mais **vous** avez dessiné chaque étape. C'est ce qui permettra d'insérer un garde-fou (Ex1.2), une validation humaine (J2) ou des sous-agents (J3) exactement là où il faut.

### [Code] Streaming token par token (`stream_mode="messages"`)

In [ ]:
with propagate_attributes(session_id="j1-03-agent", tags=["jour-1", "streaming"]):
    for morceau, meta in support.stream(
        {"messages": [HumanMessage("srv-vpn-01 est-il en ligne ?")]},
        config={"callbacks": [langfuse_handler], "run_name": "agent-streaming", "recursion_limit": 10},
        stream_mode="messages",
    ):
        if meta["langgraph_node"] == "agent" and morceau.content:   # on n'affiche que la parole de l'agent
            print(morceau.content, end="", flush=True)
print()

## 7. ⚠️ `recursion_limit` : la valeur par défaut a changé

### [Théorie] Un super-step, une marche

Chaque passage d'un nœud à l'autre compte pour une étape. Au-delà de `recursion_limit`, LangGraph lève `GraphRecursionError`. **Dans LangGraph 1.2, la valeur par défaut est 10 007** (les tutoriels annoncent encore 25, et la doc officielle affiche 1000 au 23/10/2026 : c'est le code, `langgraph/_internal/_config.py`, qui fait foi) : un agent qui boucle peut donc enchaîner des milliers d'appels LLM avant d'être arrêté.

> 🛡️ **OWASP ASI08 (Cascading Failures) :** une boucle non bornée, c'est un incident de coût et de disponibilité. **Fixez toujours `recursion_limit`** (10 à 25 pour un agent de support) et prévoyez une réponse de repli.

In [ ]:
from langgraph.errors import GraphRecursionError


class Compteur(TypedDict):
    tours: int


def relancer(state: Compteur) -> dict:
    return {"tours": state["tours"] + 1}


def toujours_relancer(state: Compteur) -> Literal["relancer"]:
    return "relancer"   # bug volontaire : aucune sortie vers END


boucle = StateGraph(Compteur)
boucle.add_node("relancer", relancer)
boucle.add_edge(START, "relancer")
boucle.add_conditional_edges("relancer", toujours_relancer)
boucle = boucle.compile()

try:
    boucle.invoke({"tours": 0}, config={"recursion_limit": 8})
except GraphRecursionError:
    # En production : journaliser, prévenir un humain, répondre poliment
    print("🛑 GraphRecursionError après 8 étapes : le graphe ne trouvait pas de chemin vers END")

## 8. Erreurs classiques avec `StateGraph`

| Symptôme | Cause | Correctif |
|---|---|---|
| L'historique disparaît à chaque tour | `messages` sans reducer | `MessagesState` ou `Annotated[list, add_messages]` |
| `InvalidUpdateError` | deux nœuds parallèles écrivent la même clé sans reducer, ou clé absente du State | déclarer la clé et son reducer |
| Le graphe ne s'arrête jamais | pas de chemin vers `END` | `tools_condition` / routeur vers `END` + `recursion_limit` |
| « Node not found » à la compilation | faute de frappe dans `add_edge` | constantes pour les noms de nœuds |
| État imprévisible | mutation sur place (`state[...] = …`) | renvoyer un nouveau `dict` partiel |
| Routage invisible dans le dessin | fonction de routage non typée | retour annoté `Literal[...]` ou `path_map` |

## 9. `create_agent` ou `StateGraph` à la main ?

| | `create_agent` | `StateGraph` explicite |
|---|---|---|
| Pour | boucle prête, middlewares (HITL, PII, limites…) | chaque étape et chaque décision sont à vous |
| Idéal | un agent autonome standard | workflows métier, garde-fous, multi-agents |
| Limite | structure imposée | plus de code à maintenir |

Les deux se combinent : un `create_agent` peut être un **nœud** d'un graphe plus large (J3).

In [ ]:
langfuse.flush()
print("✅ Observations envoyées : http://localhost:3000")

## 🧭 Récap

| Vous savez maintenant… | API |
|---|---|
| Déclarer l'état | `TypedDict`, `MessagesState` |
| Choisir la fusion des mises à jour | `Annotated[..., operator.add]`, `add_messages` |
| Câbler un graphe | `add_node`, `add_edge`, `add_conditional_edges`, `START`, `END`, `compile()` |
| Exécuter et suivre | `invoke`, `invoke(version="v2")`, `stream(stream_mode=...)` |
| Dessiner l'agent | `ToolNode`, `tools_condition` |
| Borner une boucle | `recursion_limit`, `GraphRecursionError` |

**Exercices :** `exercices/enonces/Ex1_1_stategraph_support.ipynb` puis `Ex1_2_garde_fou.ipynb`.

### ✅ Quiz express

1. Que renvoie un nœud : l'état complet ou une mise à jour ?
2. Deux nœuds parallèles écrivent `journal` sans reducer. Que se passe-t-il ?
3. À quoi sert l'annotation `Literal[...]` sur une fonction de routage ?
4. Quelle est la valeur par défaut de `recursion_limit` en LangGraph 1.2, et que faire ?

<details>
<summary>Réponses</summary>

1. Une mise à jour partielle : uniquement les clés modifiées.
2. `InvalidUpdateError` : LangGraph ne sait pas quelle valeur garder. Il faut un reducer (`operator.add`…).
3. À déclarer les destinations possibles, pour que le graphe soit dessiné (et validé) correctement.
4. 10 007. La fixer explicitement (10 à 25 pour un agent de support) et gérer `GraphRecursionError`.
</details>